# Book Recommendation System

Imagine that you have an online bookstore. There are a bunch of books rated by users. How can you build an useful recommendation system that will recommend some books to buy for users on their main page?

This notebook uses Book Crossing Dataset from Kaggle:
https://www.kaggle.com/datasets/syedjaferk/book-crossing-dataset/data?select=BX-Users.csv


# Dataset

Collected by Cai-Nicolas Ziegler in a 4-week crawl (August / September 2004) from the Book-Crossing community with kind permission from Ron Hornbaker, CTO of Humankind Systems. Contains 278,858 users (anonymized but with demographic information) providing 1,149,780 ratings (explicit / implicit) about 271,379 books.

In [193]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

The dataset is divided into three tables:

## Users

In [194]:
users_df = pd.read_csv('BX-Users.csv', sep=';', encoding='latin-1')
users_df

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN
...,...,...,...
278853,278854,"portland, oregon, usa",NaN
278854,278855,"tacoma, washington, united kingdom",50.0
278855,278856,"brampton, ontario, canada",NaN
278856,278857,"knoxville, tennessee, usa",NaN


We have 278858 users with their ages and locations. Locations are in format: city, region, country. Let's divide it into three different columns

In [195]:
location_parts = users_df['Location'].str.split(', ', expand=True)
users_df['City'] = location_parts[0]
users_df['Region'] = location_parts[1]
users_df['Country'] = location_parts[2]
users_df

,User-ID,Location,Age,City,Region,Country
0,1,"nyc, new york, usa",NaN,nyc,new york,usa
1,2,"stockton, california, usa",18.0,stockton,california,usa
2,3,"moscow, yukon territory, russia",NaN,moscow,yukon territory,russia
3,4,"porto, v.n.gaia, portugal",17.0,porto,v.n.gaia,portugal
4,5,"farnborough, hants, united kingdom",NaN,farnborough,hants,united kingdom
...,...,...,...,...,...,...
278853,278854,"portland, oregon, usa",NaN,portland,oregon,usa
278854,278855,"tacoma, washington, united kingdom",50.0,tacoma,washington,united kingdom
278855,278856,"brampton, ontario, canada",NaN,brampton,ontario,canada
278856,278857,"knoxville, tennessee, usa",NaN,knoxville,tennessee,usa


As we see, there are a lot of missing values in Age, let's check if there are a lot of other missing values

In [196]:
users_df.isna().sum()

,0
User-ID,0
Location,0
Age,110762
City,0
Region,1
Country,4577


There are not a lot of users without country so we can filter them. But almost half of users do not have assigned age, we cannot delete it so let's just replace it with mean

In [197]:
users_no_null = users_df.copy()
users_no_null['Age'].fillna(users_no_null['Age'].mean(), inplace=True)
users_no_null = users_no_null.dropna()
users_no_null

/tmp/ipython-input-1549217956.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  users_no_null['Age'].fillna(users_no_null['Age'].mean(), inplace=True)


,User-ID,Location,Age,City,Region,Country
0,1,"nyc, new york, usa",34.751434,nyc,new york,usa
1,2,"stockton, california, usa",18.000000,stockton,california,usa
2,3,"moscow, yukon territory, russia",34.751434,moscow,yukon territory,russia
3,4,"porto, v.n.gaia, portugal",17.000000,porto,v.n.gaia,portugal
4,5,"farnborough, hants, united kingdom",34.751434,farnborough,hants,united kingdom
...,...,...,...,...,...,...
278853,278854,"portland, oregon, usa",34.751434,portland,oregon,usa
278854,278855,"tacoma, washington, united kingdom",50.000000,tacoma,washington,united kingdom
278855,278856,"brampton, ontario, canada",34.751434,brampton,ontario,canada
278856,278857,"knoxville, tennessee, usa",34.751434,knoxville,tennessee,usa


But some users have unrealistic ages like 200

In [198]:
users_no_null[users_no_null['Age'] > 200]

,User-ID,Location,Age,City,Region,Country
1578,1579,"akure, ondo/nigeria, nigeria",231.0,akure,ondo/nigeria,nigeria
8457,8458,"milano, lombardia, italy",230.0,milano,lombardia,italy
8781,8782,"calgary, alberta, canada",239.0,calgary,alberta,canada
13272,13273,"harrisburg, pennsylvania, usa",201.0,harrisburg,pennsylvania,usa
20856,20857,"stuttgart, baden-wuerttemberg, germany",244.0,stuttgart,baden-wuerttemberg,germany
37834,37835,"dunbarton, new hampshire, usa",219.0,dunbarton,new hampshire,usa
52471,52472,"frisco/dallas, texas, usa",209.0,frisco/dallas,texas,usa
52648,52649,"vancouver, british columbia, canada",212.0,vancouver,british columbia,canada
58285,58286,"midvale, utah, usa",237.0,midvale,utah,usa
76864,76865,"arrasate, gipuzkoa, euskal herria",209.0,arrasate,gipuzkoa,euskal herria


We have to replace it with some max number like 100

In [199]:
users_no_null.loc[users_no_null['Age'] > 100, 'Age'] = 100
users_no_null[users_no_null['Age'] > 100]

,User-ID,Location,Age,City,Region,Country


## Books

In [200]:
books_df = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
books_df

/tmp/ipython-input-4120766662.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df = pd.read_csv('BX-Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm),http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


In this table, all the books are described. There are 271360 books. Those have attributes like:
- ISBN - unique book id
- Book-title (string)
- Book-author (string)
- Year-Of-Publication (int)
- Publisher (string)
And the images links:
- Image-URL-S
- Image-URL-M
- Image-URL-L

In [201]:
books_df.isna().sum()

,0
ISBN,0
Book-Title,0
Book-Author,2
Year-Of-Publication,0
Publisher,2
Image-URL-S,0
Image-URL-M,0
Image-URL-L,3


As we see there are NaN values in publisher, we can skip nulls in images because there won't be used

In [202]:
books_df = books_df.dropna(subset=['Publisher'])
books_df

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...
...,...,...,...,...,...,...,...,...
271355,0440400988,There's a Bat in Bunk Five,Paula Danziger,1988,Random House Childrens Pub (Mm),http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...,http://images.amazon.com/images/P/0440400988.0...
271356,0525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...,http://images.amazon.com/images/P/0525447644.0...
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...,http://images.amazon.com/images/P/006008667X.0...
271358,0192126040,Republic (World's Classics),Plato,1996,Oxford University Press,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...,http://images.amazon.com/images/P/0192126040.0...


## Books ratings

And we have a table of every rating made by the users

In [203]:
ratings_df = pd.read_csv('BX-Book-Ratings.csv', sep=';', encoding='latin-1')
ratings_df

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
...,...,...,...
1149775,276704,1563526298,9
1149776,276706,0679447156,0
1149777,276709,0515107662,10
1149778,276721,0590442449,10


As we see we have
- User-ID - foreign key referencing Users
- ISBN - foreign key referencing Books
- Book-Rating - in the scale from 0 to 10

In [204]:
ratings_df.isna().sum()

,0
User-ID,0
ISBN,0
Book-Rating,0


For the system purpose, let's normalize the ratings

In [205]:
ratings_df['Book-Rating'] = ratings_df['Book-Rating'] / 10
ratings_df

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0.0
1,276726,0155061224,0.5
2,276727,0446520802,0.0
3,276729,052165615X,0.3
4,276729,0521795028,0.6
...,...,...,...
1149775,276704,1563526298,0.9
1149776,276706,0679447156,0.0
1149777,276709,0515107662,1.0
1149778,276721,0590442449,1.0


# Brief analysis

As there are not a lot of interesting variables, let's analyze only some

In [206]:
users_df.describe()

,User-ID,Age
count,278858.00000,168096.000000
mean,139429.50000,34.751434
std,80499.51502,14.428097
min,1.00000,0.000000
25%,69715.25000,24.000000
50%,139429.50000,32.000000
75%,209143.75000,44.000000
max,278858.00000,244.000000


In [207]:
books_df.describe()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
count,271358,271358,271356,271358,271358,271358,271358,271355
unique,271358,242134,102021,202,16807,271042,271042,271039
top,0767409752,Selected Poems,Agatha Christie,2002,Harlequin,http://images.amazon.com/images/P/042509474X.0...,http://images.amazon.com/images/P/042509474X.0...,http://images.amazon.com/images/P/031228375X.0...
freq,1,27,632,13902,7535,2,2,2


In [208]:
mean_rating = ratings_df['Book-Rating'].mean()
mean_rating

np.float64(0.28669501991685364)

# Recommendation system

**WARNING**: I know I work on a very small samples of the dataset, and the results may not be satysfying, but it is caused by the google collab memory limits, it is easy to modify this notebook for bigger samples

## K-Nearest Neighbour


As we see, we have some useful data about the users and also we have their ratings of the books. Both of those are useful

Firstly lets see the most similar users to the user and recommend them the books rated the most by the most similar users

Let's only limit the searched users to users who have reviewed more than 10 books, to make the dataframe smaller

In [209]:
small_users = ratings_df['User-ID'].value_counts()
small_users = small_users[(small_users > 10) & (small_users < 100)]
small_users = pd.DataFrame(small_users)
small_users = small_users.sample(n = 1000, weights='count')
small_users = small_users.reset_index()
small_users

,User-ID,count
0,251097,47
1,108104,12
2,143497,47
3,261685,50
4,113277,32
...,...,...
995,40134,15
996,38544,66
997,1797,14
998,154529,61


Let's choose an user

In [210]:
the_user = small_users.iloc[0]['User-ID']
the_user

np.int64(251097)

And check his rated books

In [220]:
user_rated = ratings_df[ratings_df['User-ID'] == the_user]
user_rated = user_rated.merge(books_df, how='inner', on='ISBN')
user_rated[['Book-Title','Book-Author','Year-Of-Publication', 'Publisher', 'Book-Rating']]

,Book-Title,Book-Author,Year-Of-Publication,Publisher,Book-Rating
0,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,0.0
1,Travels,Michael Crichton,2002,Perennial,0.7
2,Brave New World,Aldous Huxley,1998,Perennial,0.0
3,"Plato, Not Prozac! : Applying Eternal Wisdom t...",Lou Marinoff,2000,Perennial,0.0
4,La casa de los espÃ­ritus,Isabel Allende,1995,Rayo,0.0
5,The God of Small Things,Arundhati Roy,1998,Perennial,0.0
6,The X-Files: Fight the Future,Elizabeth Hand,1998,HarperEntertainment,0.5
7,A Study in Scarlet (Sherlock Holmes),"Arthur Conan, Sir Doyle",1982,Penguin Books,0.0
8,Midnight's Children,Salman Rushdie,1995,Penguin Books,0.0
9,Before You Sleep,Linn Ullmann,2001,Penguin Books,0.0


In [212]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

In [213]:
def user_distance_books(user_id, small_users, books_df, ratings_df, minimal_score):
  merged = small_users.merge(ratings_df, how='inner', on='User-ID')
  whole = merged.merge(books_df, how='inner', on = 'ISBN')
  pivot_df = pd.pivot_table(whole, index='User-ID', columns='ISBN', values='Book-Rating')
  pivot_df.fillna(0, inplace=True)

  try:
    user_positional_index = pivot_df.index.get_loc(user_id)
  except KeyError:
    print(f"Warning: User-ID {user_id} not found in the pivot table after sampling.")
    return None, None

  pivot_sparse = csr_matrix(pivot_df.values)
  model_knn = NearestNeighbors(metric = 'cosine', algorithm = 'brute')
  model_knn.fit(pivot_sparse)
  distances, indices = model_knn.kneighbors(pivot_sparse[user_positional_index], n_neighbors = 10)
  filtered = ratings_df[ratings_df['User-ID'].isin(indices)]
  filtered = filtered[filtered['Book-Rating'] >= minimal_score]
  filtered = filtered.merge(books_df, how='inner', on='ISBN')
  return filtered.drop('Book-Rating', axis=1)

In [214]:
user_distance_books(the_user, small_users, books_df, ratings_df, 0.5)[['Book-Title','Book-Author','Year-Of-Publication', 'Publisher']]

,Book-Title,Book-Author,Year-Of-Publication,Publisher


## Matrix Factorization

Now that we've got the user-books ratings matrix, we can factorize it in order to create two matrices, one for books' hidden attributes and one for users' hidden attributes. By multiplying those two matrices we will get a similar reconstruction of our matrix

I will use the Non-Negative Matrix as our matrix has a lot of NaNs

In [215]:
def make_pivot_table(users_df, books_df, ratings_df):
  merged = users_df.merge(ratings_df, how='inner', on='User-ID')
  whole = merged.merge(books_df, how='inner', on = 'ISBN')
  pivot = pd.pivot_table(whole, index='User-ID', columns='ISBN', values='Book-Rating')
  pivot.fillna(0, inplace=True)
  return pivot

In [216]:
from sklearn.decomposition import NMF
model = NMF(n_components=5, init='random', random_state=0)
pivot = make_pivot_table(small_users, books_df, ratings_df)
W = model.fit_transform(pivot.values)
H = model.components_
reconstruct = np.dot(W, H)
print(f"Mean squared error {np.mean((pivot.values - reconstruct) ** 2)}")

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Mean squared error 0.0003738236473068574


It's small. Now let's see what books are recommended for our chosen user

In [217]:
reconstructed_table = pd.DataFrame(reconstruct, columns=pivot.columns, index=pivot.index)
user_books = ratings_df[ratings_df['User-ID'] == the_user]['ISBN'].to_numpy()
for i in user_books:
  reconstructed_table.loc[the_user, i] = 0
user_table = reconstructed_table.loc[the_user].sort_values(ascending=False)
user_table = pd.DataFrame(user_table).head(10)
user_table = user_table.reset_index()
book_recommend = user_table.merge(books_df, how='inner', on='ISBN')
book_recommend[['Book-Title','Book-Author','Year-Of-Publication', 'Publisher']]

,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,The Secret Life of Bees,Sue Monk Kidd,2003,Penguin Books
1,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,2003,Scholastic
2,The Da Vinci Code,Dan Brown,2003,Doubleday
3,The Catcher in the Rye,J.D. Salinger,1991,"Little, Brown"
4,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown"
5,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,1999,Scholastic
6,Suzanne's Diary for Nicholas,James Patterson,2001,"Little, Brown"
7,Summer's End,Danielle Steel,1980,Dell
8,Love With A Long Tall Texan,Diana Palmer,1999,Harlequin
9,His Bright Light: The Story of Nick Traina,Danielle Steel,1998,Delacorte Press


## Feature filtering for new users

But how will we recommend books for a new user? We do not have their rating history, but we have the personal data, and we can use it. Let's try to recommend books rated the best from similar users

In [218]:
def user_distance(user_id, users_df):
  distance_df = users_df.copy()
  the_user = users_df.loc[users_df['User-ID'] == user_id]

  # Extract scalar values from the_user Series for correct broadcasting
  age_scalar = the_user['Age'].iloc[0]
  city_scalar = the_user['City'].iloc[0]
  region_scalar = the_user['Region'].iloc[0]
  country_scalar = the_user['Country'].iloc[0]

  distance_df['distance'] = abs(users_df['Age'] - age_scalar) / 700 + \
                              (users_df['City'] != city_scalar) / 10 + \
                              (users_df['Region'] != region_scalar) / 10 + \
                              (users_df['Country'] != country_scalar) / 10
  return distance_df

In [219]:
distances = user_distance(the_user, users_df).sort_values('distance', ascending = True).head(100)
merged = distances.merge(ratings_df, how='inner', on='User-ID')
means = merged.groupby('ISBN')['Book-Rating'].mean().sort_values(ascending=False).head(100)
whole = merged.merge(books_df, how='inner', on='ISBN')
whole[['Book-Title','Book-Author','Year-Of-Publication', 'Publisher']]

,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,The Plague (Vintage International),ALBERT CAMUS,1991,Vintage
1,The Sound and the Fury (Vintage International),WILLIAM FAULKNER,1991,Vintage
2,Cuentos Completos/1 (Alfaguara Cuentos Completos),Julio Cortazar,1998,"Alfaguara Ediciones, S.A. (Spain)"
3,LA Sentencia (Ficcionario),Juan Carlos Botero,2003,Ediciones B
4,La divina comedia,Alighieri Dante,1999,"Editorial Edaf, S.A."
...,...,...,...,...
87,El ocho,Katherine Neville,2001,Punto de Lectura
88,Joc brut (ColÂ¨lecciÃ³ universal de butxaca el...,Manuel de Pedrolo,1976,Edicions 62
89,"L'Etranger (Collection Folio, 2)",Albert Camus,1990,Gallimard Jeunesse
90,Long Walk,Stephen King,2001,Sagebrush Bound
